# 07 · Collection and the held-out gate

Two things this suite could not do before: generate training
trajectories from the model's own attempts, and measure whether
a trained checkpoint is actually better than the stock one.

**This notebook imports the shared code instead of restating
it.** Notebooks 00-06 keep self-contained cells because each one
needs only a handful of definitions. The episode loop, the
collection filters and the scorecard are none of those: three
hand-copied copies would drift, and a gate that drifts from the
collector it grades is worse than no gate. The GPU-specific part
— turning messages into one generated turn — is the only thing
defined here.

**Held-out means held out.** The evaluation families in
`qwen3_8_27b_code.tasks` share no bug class, module or family
name with the SFT fixtures, and the test suite enforces that.
Never collect training data from them.

## Install the pinned day-zero environment

In [ ]:
import subprocess
import sys
from pathlib import Path

# The Git pins supply current Unsloth/Qwen3.8 support. Transformers, TRL and
# Datasets deliberately use the mutually compatible versions from the adjacent
# official Unsloth Qwen3.5 27B notebook. Do not replace these with branch-head
# SHAs without resolving package metadata together first.
GIT_REVISIONS = {
    "unsloth": "c87fe20e32aca9ceb2dc5059c2987738f32446e8",
    "unsloth_zoo": "5b239e574f03ab3077c17e49aeef3cacfe7cdd4e",
}

import torch

torch_version = torch.__version__.split("+", 1)[0]
torch_minor = ".".join(torch_version.split(".")[:2])
torchao_by_torch = {"2.8": "0.16.0", "2.9": "0.16.0", "2.10": "0.16.0", "2.11": "0.18.0"}
xformers_by_torch = {"2.8": "0.0.32.post2", "2.9": "0.0.33.post1", "2.10": "0.0.34", "2.11": "0.0.34"}
if torch_minor not in torchao_by_torch:
    raise RuntimeError(
        f"No reviewed Colab dependency set for torch {torch.__version__}. "
        f"Expected one of {sorted(torchao_by_torch)}; update the compatibility matrix first."
    )

COMPATIBILITY_PINS = {
    "transformers": "5.3.0",
    "trl": "0.22.2",
    "datasets": "4.3.0",
    "peft": "0.19.0",
    "torchao": torchao_by_torch[torch_minor],
    "xformers": xformers_by_torch[torch_minor],
}
INSTALLER_REVISION = "colab-v2"
pin_key = "-".join(value.replace(".", "") for value in COMPATIBILITY_PINS.values())
git_key = "-".join(value[:8] for value in GIT_REVISIONS.values())
INSTALL_KEY = f"{INSTALLER_REVISION}-torch{torch_minor}-{git_key}-{pin_key}"
INSTALL_MARKER = Path(f"/content/.qwen38_env_{INSTALL_KEY}")
PIP_LOG = Path("/content/qwen38_pip_install.log")
FORCE_INSTALL = False

def install_phase(name: str, packages: list[str], *, no_deps: bool = False) -> None:
    command = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        "--upgrade-strategy",
        "only-if-needed",
        "--no-cache-dir",
        "--log",
        str(PIP_LOG),
    ]
    if no_deps:
        command.append("--no-deps")
    command.extend(packages)
    print(f"\n=== install phase: {name} ===")
    print("\n".join(f"  {package}" for package in packages))
    result = subprocess.run(command, check=False)
    if result.returncode:
        log_tail = (
            "\n".join(PIP_LOG.read_text(errors="replace").splitlines()[-120:])
            if PIP_LOG.exists()
            else "[pip did not create its log file]"
        )
        print(f"\n--- tail of {PIP_LOG} ---\n{log_tail}")
        raise RuntimeError(
            f"Package installation failed during {name!r} with exit code {result.returncode}. "
            f"The detailed log is at {PIP_LOG}."
        )

if FORCE_INSTALL or not INSTALL_MARKER.exists():
    if PIP_LOG.exists():
        PIP_LOG.unlink()
    install_phase("packaging tools", ["pip", "setuptools==80.9.0", "wheel>=0.42.0"])
    install_phase("Qwen3.8 training stack", [
        f"unsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo.git@{GIT_REVISIONS['unsloth_zoo']}",
        f"unsloth @ git+https://github.com/unslothai/unsloth.git@{GIT_REVISIONS['unsloth']}",
        f"torch=={torch_version}",
        f"torchao=={COMPATIBILITY_PINS['torchao']}",
        f"transformers=={COMPATIBILITY_PINS['transformers']}",
        f"trl=={COMPATIBILITY_PINS['trl']}",
        f"datasets=={COMPATIBILITY_PINS['datasets']}",
        f"peft=={COMPATIBILITY_PINS['peft']}",
        "accelerate",
        "bitsandbytes",
        "trackio",
        "huggingface_hub>=0.34.0,<2.0",
        "hf_transfer",
        "sentencepiece>=0.2.0",
        "protobuf",
        "pytest",
        "jmespath",
    ])
    install_phase(
        "PyTorch-matched xFormers wheel",
        [f"xformers=={COMPATIBILITY_PINS['xformers']}"],
        no_deps=True,
    )
    INSTALL_MARKER.write_text(INSTALL_KEY)
    print("Packages installed. Restart the Colab runtime, then rerun this notebook from the top.")
else:
    print(f"Pinned environment already installed: {INSTALL_KEY}")

After the first install, restart the runtime and rerun the notebook from the top; the install marker skips the pip work.

In [ ]:
import gc
import json
import os
import platform
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

import torch
from huggingface_hub import login, whoami

if "GIT_REVISIONS" not in globals():
    raise RuntimeError(
        "This runtime was restarted. Rerun the notebook from the first cell; "
        "the install marker will skip the expensive package installation."
    )
if "COMPATIBILITY_PINS" not in globals():
    raise RuntimeError("Missing compatibility pins; rerun the notebook from the first cell.")

try:
    from google.colab import userdata
except ImportError:
    userdata = None

if not torch.cuda.is_available():
    raise RuntimeError("Select a Colab G4 GPU runtime before continuing.")

gpu = torch.cuda.get_device_properties(0)
gpu_total_gib = gpu.total_memory / 1024**3
# A vendor-labelled 96 GB card can be reported as about 89.4 GiB because
# PyTorch converts the byte count with a binary divisor. Keep the floor well
# above the roughly 44.7 GiB reported for a 48 GB card without rejecting G4.
MIN_G4_TOTAL_GIB = 85.0
print(
    f"GPU: {gpu.name} ({gpu_total_gib:.1f} GiB total), "
    f"capability={torch.cuda.get_device_capability(0)}"
)
if gpu_total_gib < MIN_G4_TOTAL_GIB:
    raise RuntimeError(
        "This suite expects the nominal 96 GB Colab G4 runtime. "
        f"PyTorch reports {gpu_total_gib:.1f} GiB total; expected at least "
        f"{MIN_G4_TOTAL_GIB:.0f} GiB. A value near 45 GiB usually indicates "
        "the 48 GB GPU variant."
    )

# IPython stores the last exception on sys.last_traceback, whose frames keep
# every local alive, including a ~52 GiB model from a failed cell. gc.collect()
# cannot free what those frames still reference.
def release_stale_gpu_state() -> float:
    for _stale_name in ("model", "tokenizer", "processor", "trainer"):
        globals().pop(_stale_name, None)
    for _exc_attr in ("last_traceback", "last_value", "last_type", "last_exc"):
        if hasattr(sys, _exc_attr):
            delattr(sys, _exc_attr)
    gc.collect()
    torch.cuda.empty_cache()
    try:
        torch._dynamo.reset()
    except AttributeError:
        pass
    return torch.cuda.mem_get_info()[0] / 1024**3

# Fail before a model load that accelerate would silently offload.
def require_free_vram(minimum_gib: float) -> float:
    free_gib = release_stale_gpu_state()
    if free_gib < minimum_gib:
        raise RuntimeError(
            f"Only {free_gib:.1f} GiB VRAM is free but this load needs about "
            f"{minimum_gib:.0f} GiB. A previous model in this kernel is still "
            "holding memory. Restart the runtime and rerun from the top."
        )
    return free_gib

# Reject a load that accelerate quietly spilled to CPU or disk. A partially
# offloaded model copies weights back per forward pass (the 2.4 GiB embedding
# alone) and is guaranteed to OOM or crawl mid-episode.
def assert_model_fully_resident(model, minimum_free_gib: float = 4.0) -> None:
    non_cuda = sorted({
        parameter.device.type
        for parameter in model.parameters()
        if parameter.device.type != "cuda"
    })
    offload_hooks = [
        name for name, module in model.named_modules()
        if getattr(getattr(module, "_hf_hook", None), "offload", False)
    ]
    if non_cuda or offload_hooks:
        raise RuntimeError(
            "The checkpoint did not fit on the GPU and accelerate offloaded "
            f"part of it (devices={non_cuda}, offload_hooks={len(offload_hooks)}). "
            "Restart the runtime to release stale VRAM, then rerun from the top."
        )
    free_gib = torch.cuda.mem_get_info()[0] / 1024**3
    if free_gib < minimum_free_gib:
        raise RuntimeError(
            f"Only {free_gib:.1f} GiB VRAM is free after the load; the KV "
            "cache and generation workspaces need headroom. Restart the "
            "runtime and rerun from the top."
        )
    print(f"Model fully resident on GPU; {free_gib:.1f} GiB VRAM free.")

release_stale_gpu_state()

hf_token = userdata.get("HF_TOKEN") if userdata is not None else os.getenv("HF_TOKEN")
if not hf_token:
    raise RuntimeError("Add a write-capable HF_TOKEN to Colab Secrets before continuing.")
login(token=hf_token, add_to_git_credential=False)
HF_USERNAME = whoami()["name"]

def package_version(name: str) -> str:
    try:
        return version(name)
    except PackageNotFoundError:
        return "missing"

observed_pins = {name: package_version(name) for name in COMPATIBILITY_PINS}
pin_mismatches = {
    name: {"expected": expected, "observed": observed_pins[name]}
    for name, expected in COMPATIBILITY_PINS.items()
    if observed_pins[name] != expected
}
if pin_mismatches:
    raise RuntimeError(
        "The runtime does not match the reviewed compatibility set. "
        f"Rerun the install cell with FORCE_INSTALL=True: {pin_mismatches}"
    )

RUN_ROOT = Path("/content/qwen38_runs")
RUN_ROOT.mkdir(parents=True, exist_ok=True)
runtime_manifest = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "gpu": gpu.name,
    "gpu_total_gib": round(gpu_total_gib, 2),
    "packages": {
        name: package_version(name)
        for name in ["unsloth", "unsloth_zoo", "transformers", "trl", "peft", "datasets"]
    },
    "git_revisions": GIT_REVISIONS,
    "compatibility_pins": COMPATIBILITY_PINS,
}
(RUN_ROOT / "runtime_manifest.json").write_text(json.dumps(runtime_manifest, indent=2))
print(json.dumps(runtime_manifest, indent=2))
print(f"Authenticated as {HF_USERNAME}")

## Bring in the shared harness, collector and gate

In [ ]:
import subprocess

REPO_URL = "https://github.com/CodeHalwell/qwen3.8-27B-code"
REPO_REVISION = "main"  # Pin an immutable commit before a run that produces artifacts.
REPO_DIR = Path("/content/qwen3.8-27B-code")

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_REVISION, REPO_URL, str(REPO_DIR)],
        check=True,
    )
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

from qwen3_8_27b_code.collection import collect, write_corpus
from qwen3_8_27b_code.episodes import EpisodeBudget, TurnResult
from qwen3_8_27b_code.evaluation import (
    compare,
    evaluate,
    gate,
    gate_passed,
    read_report,
    write_report,
)
from qwen3_8_27b_code.fixtures import iter_tasks
from qwen3_8_27b_code.tasks import evaluation_tasks, task_from_fixture

repo_revision = subprocess.run(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
    text=True, capture_output=True, check=True,
).stdout.strip()
print(json.dumps({"repo": REPO_URL, "revision": repo_revision}, indent=2))

## Run configuration

In [ ]:
from unsloth import FastLanguageModel

MODEL_ID = "unsloth/Qwen3.8-27B"
ACCEPTED_ADAPTER_ID = f"{HF_USERNAME}/qwen38-27b-code-sft-lora"
ACCEPTED_REVISION = "REPLACE_WITH_ACCEPTED_COMMIT"
MAX_SEQUENCE_LENGTH = 16_384
MAX_NEW_TOKENS_PER_TURN = 2_048
REASONING_EFFORT = "medium"

# docs/evaluation.md funnel: the sentinel tier is the cheap one
# every candidate runs. Widen only for a candidate or release
# gate, and price it before starting.
EVAL_VARIANTS_PER_FAMILY = 1     # 6 held-out tasks
EVAL_ATTEMPTS = 1                # deterministic sentinel pass
EPISODE_BUDGET = EpisodeBudget(tool_calls=10, wall_seconds=480.0)

RUN_BASELINE_EVAL = True
RUN_CANDIDATE_EVAL = False       # needs an accepted adapter revision
RUN_COLLECTION = False           # expensive; read the cost note below first
PUSH_ARTIFACTS = False

COLLECTION_ATTEMPTS = 3
COLLECTION_VARIANTS_PER_FAMILY = 2
COLLECTION_SEEDS = (3407, 9176, 20261)

REPORT_DIR = RUN_ROOT / "gate"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

if RUN_CANDIDATE_EVAL and ACCEPTED_REVISION.startswith("REPLACE_"):
    raise RuntimeError("Pin the accepted adapter revision before evaluating it.")

evaluation_suite = evaluation_tasks(variants_per_family=EVAL_VARIANTS_PER_FAMILY)
print(json.dumps({
    "held_out_tasks": len(evaluation_suite),
    "families": sorted({task.family for task in evaluation_suite}),
    "attempts_each": EVAL_ATTEMPTS,
}, indent=2))

## The only GPU-specific piece: messages in, one turn out

In [ ]:
# Everything else in this notebook is shared code. A policy is a
# callable that renders the history, generates one assistant
# turn, and reports whether generation finished or was cut off.
def build_policy_factory(model, tokenizer, reasoning_effort=REASONING_EFFORT):
    generation_eos = model.generation_config.eos_token_id
    eos_token_ids = {
        token_id
        for token_id in (
            *(generation_eos if isinstance(generation_eos, (list, tuple)) else [generation_eos]),
            tokenizer.eos_token_id,
        )
        if token_id is not None
    }
    if not eos_token_ids:
        raise RuntimeError("No end-of-turn token id is available; truncation cannot be detected.")

    def policy_factory(task, seed):
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

        def policy(messages):
            rendered = render_chat(
                messages,
                add_generation_prompt=True,
                reasoning_effort=reasoning_effort,
            )
            inputs = tokenizer(
                text=rendered, return_tensors="pt", add_special_tokens=False
            ).to("cuda")
            prompt_tokens = int(inputs["input_ids"].numel())
            if prompt_tokens + MAX_NEW_TOKENS_PER_TURN > MAX_SEQUENCE_LENGTH:
                return TurnResult(
                    text="", prompt_tokens=prompt_tokens, fault="context_budget"
                )
            with torch.inference_mode():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=MAX_NEW_TOKENS_PER_TURN,
                    temperature=1.0,
                    top_p=0.95,
                    top_k=20,
                    do_sample=True,
                    use_cache=True,
                )
            new_ids = outputs[0, inputs["input_ids"].shape[1]:]
            completion_tokens = int(new_ids.numel())
            stopped_on_eos = completion_tokens > 0 and int(new_ids[-1]) in eos_token_ids
            return TurnResult(
                text=tokenizer.decode(new_ids, skip_special_tokens=False),
                prompt_tokens=prompt_tokens,
                completion_tokens=completion_tokens,
                fault=None if stopped_on_eos else "output_truncated",
            )

        return policy

    return policy_factory

In [ ]:
# Bumped from v1 when the `shell` description stopped carrying pilot status
# text. Tool descriptions are model inputs and part of the fingerprint, so a
# wording change is a schema change.
TOOL_SCHEMA_VERSION = "qwen38-six-tools-v2"

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "list_files",
            "description": "List files below a repository-relative directory.",
            "parameters": {
                "type": "object",
                "properties": {"path": {"type": "string"}},
                "required": ["path"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Read a UTF-8 repository file with bounded output.",
            "parameters": {
                "type": "object",
                "properties": {"path": {"type": "string"}},
                "required": ["path"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "search",
            "description": "Search repository text using a regular expression.",
            "parameters": {
                "type": "object",
                "properties": {"query": {"type": "string"}},
                "required": ["query"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "apply_patch",
            "description": "Apply a unified diff to files inside the repository.",
            "parameters": {
                "type": "object",
                "properties": {"patch": {"type": "string"}},
                "required": ["patch"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "run_tests",
            "description": "Run an allow-listed repository test profile.",
            "parameters": {
                "type": "object",
                "properties": {"profile": {"type": "string", "enum": ["unit"]}},
                "required": ["profile"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "shell",
            "description": "Run a command from the harness allow-list.",
            "parameters": {
                "type": "object",
                "properties": {"command": {"type": "string"}},
                "required": ["command"],
                "additionalProperties": False,
            },
        },
    },
]

def _without_arrow_nulls(value):
    """Remove null struct fields inserted by a Datasets/Arrow round trip."""
    if isinstance(value, dict):
        cleaned = {}
        for key, item in value.items():
            normalized = _without_arrow_nulls(item)
            if normalized is not None:
                cleaned[key] = normalized
        return cleaned
    if isinstance(value, list):
        return [_without_arrow_nulls(item) for item in value]
    return value

def canonical_tool_schema(tools: list[dict]) -> str:
    """Return a stable semantic fingerprint while retaining tool order."""
    return json.dumps(
        _without_arrow_nulls(tools),
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
    )

TOOL_SCHEMA_JSON = canonical_tool_schema(TOOLS)

def rendered_tool_schema(rendered_prompt: str) -> str:
    """Extract and canonicalise JSON tool declarations from a Qwen prompt."""
    start_tag = "<tools>"
    end_tag = "</tools>"
    if start_tag not in rendered_prompt or end_tag not in rendered_prompt:
        raise ValueError("Rendered prompt does not contain a <tools> block.")
    payload = rendered_prompt.split(start_tag, 1)[1].split(end_tag, 1)[0]
    try:
        rendered_tools = [
            json.loads(line)
            for line in payload.splitlines()
            if line.strip()
        ]
    except json.JSONDecodeError as exc:
        raise ValueError("Rendered <tools> block is not newline-delimited JSON.") from exc
    return canonical_tool_schema(rendered_tools)

def canonical_to_qwen(messages: list[dict]) -> list[dict]:
    """Merge the leading policy messages into one system message.

    The Qwen3.8 template accepts `developer` natively and merges a run of
    leading system/developer messages itself. This fold is therefore not a
    compatibility shim: it exists so training and deployment both hand the
    template one deterministically joined policy message.
    """
    converted = []
    pending_system = []
    for stored_message in messages:
        message = _without_arrow_nulls(stored_message)
        role = message["role"]
        if role in {"system", "developer"} and not converted:
            pending_system.append(str(message.get("content", "")))
            continue
        if pending_system:
            converted.append({"role": "system", "content": "\n\n".join(pending_system)})
            pending_system = []
        converted.append(message)
    if pending_system:
        converted.append({"role": "system", "content": "\n\n".join(pending_system)})
    return converted

def render_chat(messages: list[dict], *, add_generation_prompt: bool, reasoning_effort: str = "medium") -> str:
    return tokenizer.apply_chat_template(
        canonical_to_qwen(messages),
        tools=TOOLS,
        tokenize=False,
        add_generation_prompt=add_generation_prompt,
        enable_thinking=True,
        reasoning_effort=reasoning_effort,
        preserve_thinking=True,
    )

## Baseline: the stock model on the held-out suite

In [ ]:
baseline_report_path = REPORT_DIR / "baseline.json"

if RUN_BASELINE_EVAL:
    require_free_vram(60.0)
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_ID,
        max_seq_length=MAX_SEQUENCE_LENGTH,
        load_in_4bit=False,
        full_finetuning=False,
        token=hf_token,
    )
    assert_model_fully_resident(model)
    FastLanguageModel.for_inference(model)

    baseline = evaluate(
        evaluation_suite,
        build_policy_factory(model, tokenizer),
        label="upstream-bf16",
        attempts_per_task=EVAL_ATTEMPTS,
        budget=EPISODE_BUDGET,
    )
    write_report(baseline, baseline_report_path)
    print(json.dumps(baseline.scorecard(), indent=2))
    print(f"wrote {baseline_report_path}")
else:
    print("Baseline evaluation is off. It is the comparison point for every later claim.")

## Candidate: the accepted adapter on the same frozen suite

Release the baseline model first. Two 27B checkpoints do not
coexist on one card, and a partially offloaded second load
crawls or dies mid-episode.

In [ ]:
candidate_report_path = REPORT_DIR / "candidate.json"

if RUN_CANDIDATE_EVAL:
    release_stale_gpu_state()
    require_free_vram(60.0)
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=ACCEPTED_ADAPTER_ID,
        revision=ACCEPTED_REVISION,
        max_seq_length=MAX_SEQUENCE_LENGTH,
        load_in_4bit=False,
        token=hf_token,
    )
    assert_model_fully_resident(model)
    FastLanguageModel.for_inference(model)

    candidate = evaluate(
        evaluation_suite,
        build_policy_factory(model, tokenizer),
        label="sft-lora-candidate",
        attempts_per_task=EVAL_ATTEMPTS,
        budget=EPISODE_BUDGET,
    )
    write_report(candidate, candidate_report_path)
    print(json.dumps(candidate.scorecard(), indent=2))
else:
    print("Candidate evaluation is off. Turn it on once an adapter revision is accepted.")

## Apply the gate

In [ ]:
if baseline_report_path.exists() and candidate_report_path.exists():
    comparison = compare(
        read_report(baseline_report_path), read_report(candidate_report_path)
    )
    checks = gate(comparison)
    comparison["gate"] = [
        {"name": check.name, "passed": check.passed, "detail": check.detail}
        for check in checks
    ]
    comparison["gate_passed"] = gate_passed(checks)
    (REPORT_DIR / "comparison.json").write_text(json.dumps(comparison, indent=2))

    print(json.dumps(comparison["deltas"], indent=2))
    for check in checks:
        print(f"  [{'PASS' if check.passed else 'FAIL'}] {check.name}: {check.detail}")
    task_level = comparison["task_level"]
    # A suite this small reports paired outcomes; it cannot
    # support a percentage-point significance claim.
    print(
        f"{task_level['wins']} improved, {task_level['losses']} regressed, "
        f"{task_level['ties']} unchanged, of {task_level['tasks']} tasks."
    )
    print("GATE PASSED" if comparison["gate_passed"] else "GATE FAILED")
else:
    print("Both a baseline and a candidate report are required before the gate can run.")

## Collect training trajectories by rejection sampling

This is the route off the scripted bootstrap corpus. The model
attempts each training task several times, every attempt is
graded from outside its workspace, and only verified attempts
become rows — carrying the model's own reasoning at the effort
it ran at, which is what the scripted corpus cannot supply.

Cost first: attempts × tasks × mean episode seconds. Measure one
task before enabling the full sweep, and use the acceptance rate
in the report to decide whether more attempts or easier tasks
are the better next move.

In [ ]:
if RUN_COLLECTION:
    if "model" not in globals():
        raise RuntimeError("Load a model in one of the cells above before collecting.")
    collection_tasks = [
        task_from_fixture(fixture)
        for fixture in iter_tasks(COLLECTION_VARIANTS_PER_FAMILY)
    ]
    result = collect(
        collection_tasks,
        build_policy_factory(model, tokenizer),
        attempts_per_task=COLLECTION_ATTEMPTS,
        seeds=COLLECTION_SEEDS,
        budget=EPISODE_BUDGET,
        reasoning_effort=REASONING_EFFORT,
        max_rows_per_task=2,
    )
    corpus_path = REPORT_DIR / "collected_trajectories.jsonl"
    report = write_corpus(result, corpus_path, REPORT_DIR / "collection_report.json")
    print(json.dumps(report, indent=2))
    print(f"wrote {corpus_path}; feed it to notebook 02 as SOURCE_LOCAL_JSONL.")

    if PUSH_ARTIFACTS:
        from huggingface_hub import HfApi

        HfApi(token=hf_token).upload_folder(
            repo_id=f"{HF_USERNAME}/qwen38-code-collected-v0",
            repo_type="dataset",
            folder_path=str(REPORT_DIR),
            private=True,
        )
else:
    print("Collection is off. Enable it once the baseline scorecard shows the failure mix.")

## What the numbers mean

Read the acceptance rate and the rejection breakdown before the
row count. A corpus of 500 rows whose rejections are dominated
by `completed_without_verification` is telling you the policy
does not verify, and training on the survivors will not fix that.

The difficulty bands come from docs/data-strategy.md: tasks in
the trivial band are protocol smoke tests, the learnable band is
the useful curriculum, and frontier tasks are for later.

Feed the collected JSONL to notebook 02, which remains the
publisher that validates, splits and pushes the dataset that
notebooks 03 and 06 consume.